In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = "workspace.drift_monitor.silver_transactions"
GOLD_DAILY = "workspace.drift_monitor.gold_daily_summary"

In [0]:
gold_daily = (
    spark.table(SILVER_TABLE)
    .groupBy("day", "transaction_type")
    .agg(
        F.count("*").alias("txns"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount"),
        F.sum(F.col("is_fraud").cast("int")).alias("fraud_txns"),
    )
)

gold_daily.write.mode("overwrite").saveAsTable(GOLD_DAILY)

Reconcile

In [0]:
%sql
SELECT
    COUNT(*)        AS gold_rows,
    SUM(txns)       AS total_txns,
    SUM(fraud_txns) AS total_fraud
FROM workspace.drift_monitor.gold_daily_summary

In [0]:
%sql
WITH daily AS (
    SELECT
        day,
        SUM(txns)                                      AS txns,
        SUM(fraud_txns)                                AS fraud_txns,
        ROUND(100.0 * SUM(fraud_txns) / SUM(txns), 3)  AS fraud_pct
    FROM workspace.drift_monitor.gold_daily_summary
    GROUP BY day
)
SELECT
    day, txns, fraud_txns, fraud_pct,
    ROUND(fraud_pct - LAG(fraud_pct) OVER (ORDER BY day), 3) AS change_vs_prev_day
FROM daily
ORDER BY day

In [0]:
%sql
SELECT day, COUNT(*) AS types_present
FROM workspace.drift_monitor.gold_daily_summary
GROUP BY day
HAVING COUNT(*) < 5
ORDER BY day

In [0]:
%sql
SELECT transaction_type, txns, fraud_txns,
       ROUND(100.0 * fraud_txns / txns, 1) AS fraud_pct
FROM workspace.drift_monitor.gold_daily_summary
WHERE day = 31

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.drift_monitor.results;